In [ ]:
import os.path
from spaceKLIP import database, imagetools,pyklippipeline
import numpy as np
import shutil
import glob

# Setting up directories

In [ ]:

# Output directory
output_dir = '/path/to/outdir'

# Science files that you want to use the PSF library on.
# They need to be at the "nanreplaced" step of the spaceKLIP tutorial pipeline 
# (i.e., after all the bad pixel cleaning, but before any shifting or re-centering)
nan_replaced_science_files = glob.glob('/path/to/nanreplaced_sciencefiles/*calints.fits')

# The actual PSF library files.
# As uploaded in the Box, the are at the "nanreplaced" step of the spaceKLIP tutorial pipeline 
# (i.e., after all the bad pixel cleaning, but before any shifting or re-centering)
nan_replaced_reference_files = glob.glob('/path/to/PSFLibrary_nanreplaced_offaxismasked/*calints.fits')

# If there are extra nanreplaced files to include (e.g., observations from the program you are reducing), add them here.
# They need to be at the "nanreplaced" step of the spaceKLIP tutorial pipeline 
# (i.e., after all the bad pixel cleaning, but before any shifting or re-centering)
nan_replaced_reference_files+= glob.glob('/path/to/nanreplaced_extrareferences/*calints.fits')

# Set up database and imagetools object

In [ ]:
Database = database.Database(output_dir=output_dir)

Database.read_jwst_s012_data(datapaths=nan_replaced_science_files,
                        psflibpaths=nan_replaced_reference_files,
                        bgpaths=None)
ImageTools = imagetools.ImageTools(Database)

# Rest of spaceKLIP pipeline (here showing what was done for the CATNIP KLIP subtractions)

In [ ]:
ImageTools.update_nircam_centers()

In [ ]:
# Set up spectral type of the science target being KLIP'd
sptype = 'M3' # example

ImageTools.calculate_centers(spectral_type=sptype, subdir='recenter_shifts',
                             use_ta=False, first_sci_only=True)

In [ ]:
# Only run this cell if first_sci_only=True above (recommended). See spaceKLIP tutorial for details
ImageTools.calculate_alignment(method='fourier', subdir='alignment_shifts')

In [ ]:
ImageTools.shift_frames(method='fourier', subdir='shifted')

In [ ]:
# This step is not in the spaceKLIP tutorial as of the time of uploading this notebook,
# but it is recommended! Make sure you have the latest spaceKLIP version for this
ImageTools.subtract_nircam_coron_background()

In [ ]:
# This step is not in the spaceKLIP tutorial as of the time of uploading this notebook,
# but it is recommended! Make sure you have the latest spaceKLIP version for this
ImageTools.mask_NDsquares(npix=2)

In [ ]:
# Optional: padding frames
ImageTools.pad_frames(npix=40, types=['SCI', 'SCI_BG', 'REF', 'REF_BG'], cval=np.nan)

In [ ]:
# Optional: using the PSF library creates a LOT of data products so
# storage can be an issue. In practice it's usually enough to just keep the 
# last products of the pipeline RIGHT BEFORE doing PSF subtraction
# for us to do e.g. companion fitting, calibration of contrast curves, or just do KLIP 
# many different times.

# So, optionally, we can delete all data products except for the last ones (in this case, the 'padded' files)

# Removing unnecessary files (if storage is limited)
shutil.rmtree(os.path.join(output_dir,'bgsub'))
shutil.rmtree(os.path.join(output_dir,'alignment_shifts'))
shutil.rmtree(os.path.join(output_dir,'recenter_shifts'))
shutil.rmtree(os.path.join(output_dir,'shifted'))
shutil.rmtree(os.path.join(output_dir,'ndmasked'))

In [ ]:
# KLIP subtraction
pyklippipeline.run_obs(database=Database,
                            kwargs={'mode': [ 'RDI' ],
                                    'annuli': [ 1,3 ],
                                    'subsections': [ 1 ],
                                    'maxnumbasis': None,
                                    'numbasis': [ 1, 2,3,4,5,10,20,40,60,80,100,150,200,250,300,350,400,450,500,600,700,800,900,1000], # there's a lot of frames now with the PSF library
                                    'algo': 'klip' ,
                                    'save_full_output': True, # New option in the latest version of spaceKLIP
                                    'save_rolls': True},
                            subdir='klipsub')